In [1]:
import jax

jax.config.update("jax_enable_x64", True)
import equinox as eqx
import jax_sn 
import jax_sbgeom as jsb
import jax.numpy as jnp
import os 
import h5py
import matplotlib.pyplot as plt

In [2]:
vmec_file = ["/home/tbogaarts/stellarator_paper/base_data/vmecs/helias5_vmec.nc4"][0]
fs_base = jsb.flux_surfaces.FluxSurfaceNormalExtendedNoPhi.from_hdf5(vmec_file)

coils_file = ['/home/tbogaarts/stellarator_paper/base_data/vmecs/HELIAS5_coils_all.h5'][0]

W0302 15:28:32.661264   36301 cuda_executor.cc:1802] GPU interconnect information not available: INTERNAL: NVML doesn't support extracting fabric info or NVLink is not used by the device.
W0302 15:28:32.665062   36257 cuda_executor.cc:1802] GPU interconnect information not available: INTERNAL: NVML doesn't support extracting fabric info or NVLink is not used by the device.


In [3]:
def save_dict(d, filename):
    print("Saving ",d , " to " ,filename)
    import pickle
    with open(filename, "wb") as f:
        pickle.dump(d, f)


theta         =  jnp.pi/25
theta_inboard = jnp.pi * 9/8.0
sigma_inboard = 0.5

In [92]:
with h5py.File(coils_file, "r") as f:
    coil_positions = jnp.array(f['Dataset1'])


coils = jsb.coils.CoilSet(coils=jsb.coils.DiscreteCoil(coil_positions, jnp.mean(coil_positions,axis=1)))

fourier_coilset = jsb.coils.convert_to_fourier_coilset(coils)
d_edge_norm = 1.8

In [93]:
n_points_per_coil = 100 
n_points_toroidal = 500 

cws_optimized = jsb.coils.create_optimized_coil_winding_surface(fourier_coilset, n_points_per_coil, n_points_toroidal, surface_type="spline")

In [94]:
flux_surface_extended = jsb.flux_surfaces.create_extended_flux_surface_d_interp_equal_arclength(fs_base, d_edge_norm, 70, 80, 100)    

In [109]:
def get_available_distance(n_theta, phi, d_value_sample):
    
    theta_sample = jnp.linspace(0, 2 * jnp.pi, n_theta, endpoint=False)

    phi_sample = jnp.array([phi])

    position_sample = flux_surface_extended.cartesian_position(1.0 + (d_value_sample)/d_edge_norm, theta_sample, phi_sample)
    directions_lcfs = (position_sample - flux_surface_extended.cartesian_position(1.0, theta_sample, phi_sample))
    directions_lcfs = directions_lcfs / jnp.linalg.norm(directions_lcfs, axis=-1, keepdims=True)

    dlcfs = jsb.jax_utils.raytracing.find_minimum_distance_to_mesh(position_sample, directions_lcfs, cws_optimized) + d_value_sample

    return dlcfs

In [115]:
n_phi = 30


n_theta_pol_slice = 50


In [117]:
phi_samples = jnp.linspace(0, 2 * jnp.pi / flux_surface_extended.settings.nfp / 2.0, n_phi, endpoint=True)
dlcfs_matrix = jax.vmap(get_available_distance, in_axes=(None, 0, None))(n_theta_pol_slice, phi_samples, 1.6)

In [121]:
jnp.save("dlcfs_matrix.npy", dlcfs_matrix)
jnp.save("phi_samples.npy", phi_samples)